# Benchmark détection + modélisation des chirps

Benchmark reproductible contre `bat_chirp_annotations.json`. Le mode du détecteur est **explicite** afin d'éviter de relancer accidentellement le legacy.

Modes disponibles dans `bat_analysis/detection.py` : `legacy`, `adaptive_v2`, `adaptive_v3`. `adaptive_v3` reste expérimental tant qu'il n'a pas été validé sur davantage de WAV et des fichiers `no_chirp`.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from tkinter import Tk, filedialog
import pandas as pd

from benchmark import run_benchmark
from benchmark.diagnostics import print_modelling_diagnostics


## Projet et mode à tester

In [ ]:
def find_project_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'bat_analysis').exists():
            return candidate
    raise FileNotFoundError('Impossible de trouver la racine du projet')

PROJECT_ROOT = find_project_root()
analysis_py = PROJECT_ROOT / 'bat_analysis' / 'modelling.py'
DETECTOR_MODE = 'adaptive_v3'   # 'legacy', 'adaptive_v2' ou 'adaptive_v3'

print('Projet     :', PROJECT_ROOT)
print('Traitement :', analysis_py)
print('Détecteur  :', DETECTOR_MODE)


## Choisir le JSON d'annotation

In [ ]:
root = Tk()
root.withdraw()
root.attributes('-topmost', True)
annotation_json = filedialog.askopenfilename(
    title='Sélectionner bat_chirp_annotations.json',
    filetypes=[('JSON', '*.json'), ('Tous les fichiers', '*.*')],
)
root.destroy()
annotation_json = Path(annotation_json)
print('Annotations :', annotation_json)


## Lancer le benchmark complet

Le modeller reçoit automatiquement `peak_freq_hz` du candidat lorsque le module courant le supporte.

In [ ]:
result = run_benchmark(
    annotation_json=annotation_json,
    analysis_py=analysis_py,
    detector_kwargs={'slope_filter_mode': DETECTOR_MODE},
    min_iou=0.05,
    max_center_error_ms=4.0,
    verbose=True,
)


## Résumé global

In [ ]:
pd.Series(result.summary)


## Résultats par WAV

In [ ]:
result.files.sort_values(['fn', 'fp'], ascending=False)


## FN restants

In [ ]:
fn = result.chirps[result.chirps['detected'] == False].copy()
print(f'{len(fn)} FN')
fn[['relative_path', 'chirp_id', 'manual_start_ms', 'manual_end_ms', 'manual_duration_ms']]


## FP restants

In [ ]:
fp = result.detections[result.detections['matched'] == False].copy()
print(f'{len(fp)} FP')
fp.sort_values(['relative_path', 'time_mid_ms'])


## Comparaison aux références de développement

In [ ]:
references = pd.DataFrame({
    'legacy': {'true_positives':113, 'false_positives':28, 'false_negatives':45, 'precision':0.801418, 'recall':0.715190, 'f1':0.755853},
    'adaptive_v2': {'true_positives':145, 'false_positives':25, 'false_negatives':13, 'precision':0.852941, 'recall':0.917722, 'f1':0.884146},
})
current = pd.Series({k: result.summary.get(k) for k in references.index}, name=DETECTOR_MODE)
comparison = references.copy()
comparison[DETECTOR_MODE] = current
comparison


## Diagnostic de modélisation

In [ ]:
diag = print_modelling_diagnostics(result)


## Sauvegarder les résultats

In [ ]:
output_dir = Path(annotation_json).parent / 'benchmark_results' / DETECTOR_MODE
result.save_csv(output_dir)
comparison.to_csv(output_dir / 'comparison_detector_modes.csv')
print('Résultats sauvegardés dans :', output_dir)
